##### Configuration

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

sys.path.insert(0, '../src')
sys.path.insert(0, '..')

%load_ext autoreload
%autoreload 2
import query_model as query_model

c:\Users\admin\miniforge3\envs\Brain\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
MODEL_PATHS = [
    # 'qwen/qwen-2.5-7b-instruct',
    # 'qwen/qwen-2.5-14b-instruct',
    # 'qwen/qwen-2.5-32b-instruct',
    # 'qwen/qwen-2.5-72b-instruct',

    # 'qwen/qwen3-8b',
    # 'qwen/qwen3-14b',
    # 'qwen/qwen3-32b',
    # 'qwen/qwen3-235b-a22b',

    # 'google/gemma-2-9b-it',
    # 'google/gemma-2-27b-it',

    # 'google/gemma-3-4b-it',
    # 'google/gemma-3-12b-it',
    'google/gemma-3-27b-it',

    # 'allenai/olmo-3-7b-instruct',
    # 'allenai/olmo-3.1-32b-instruct',

    # 'openai/gpt-oss-20b',
    # 'openai/gpt-oss-120b',

    'mistralai/mistral-small-24b-instruct-2501',
    # 'mistralai/mistral-small-3.2-24b-instruct',
    # 'mistralai/mistral-small-2603',
    # 'mistralai/mistral-medium-3.1',
    # 'mistralai/mistral-large-2512',

    # 'z-ai/glm-4.5-air',
    # 'z-ai/glm-4.5',

    # 'moonshotai/kimi-k2.5',

    # 'openai/gpt-5.4-nano',
    # 'openai/gpt-5.4-mini',
    # 'openai/gpt-5.4',

    # 'anthropic/claude-sonnet-4.6',
    # 'anthropic/claude-opus-4.6',

    # 'google/gemini-2.5-flash-lite',
    # 'google/gemini-2.5-flash',
    # 'google/gemini-2.5-pro',
]

MODEL_PATH = MODEL_PATHS[0]
MODEL_NAME = MODEL_PATH.split('/')[-1]

os.makedirs(f'./figures/{MODEL_NAME}', exist_ok=True)

N_ITERS = 100

##### OpenRouter client

In [3]:
from env_keys import load_env  # src/env_keys.py
load_env()  # API keys from the repo-root .env; never hardcode them here

from openai import OpenAI

client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=os.environ['OPENROUTER_API_KEY'],
)

print(f'OpenRouter API key set; client ready for {MODEL_PATH}')

OpenRouter API key set; client ready for google/gemma-3-27b-it


##### Load stimuli

In [4]:
stim_path = '../stimuli/'

grammar_items = pd.read_csv(os.path.join(stim_path, 'MoralGrammar68.csv'))
economic_items = pd.read_csv(os.path.join(stim_path, 'MoralEconomic68.csv'))
human_ratings = pd.read_json(os.path.join(stim_path, 'moralgrammar68_human_ratings.json'), lines=True)

print(f'MoralGrammar68: {grammar_items.shape}')
print(f'MoralEconomic68: {economic_items.shape}')
print(f'Human ratings: {human_ratings.shape}')

MoralGrammar68: (68, 3)
MoralEconomic68: (68, 7)
Human ratings: (68, 7)


##### Generate / load ratings

In [5]:
# ── Generate ratings for all models ──
import traceback

errored = []
for mp in MODEL_PATHS:
    mn = mp.split('/')[-1]
    print(f'\n{"=" * 70}\n[{mn}]\n{"=" * 70}')
    os.makedirs(f'./data/{mn}', exist_ok=True)
    grammar_file = f'./data/{mn}/moralgrammar68_ratings.csv'
    economic_file = f'./data/{mn}/moraleconomic68_ratings.csv'

    if os.path.exists(grammar_file) and os.path.exists(economic_file):
        print(f'[{mn}] cached, skipping.')
        continue

    try:
        if not os.path.exists(grammar_file):
            df = query_model.run_ratings_iter(
                stim_list=grammar_items['item'].tolist(),
                iters=N_ITERS, secondary_type='grammar',
                model_name=mp,
            )
            df.to_csv(grammar_file, index=False)
        if not os.path.exists(economic_file):
            df = query_model.run_ratings_iter(
                stim_list=economic_items['item'].tolist(),
                iters=N_ITERS, secondary_type='economic',
                model_name=mp,
            )
            df.to_csv(economic_file, index=False)
    except Exception as e:
        print(f'\n[{mn}] FAILED:')
        traceback.print_exc()
        errored.append((mn, type(e).__name__, str(e)))

if errored:
    print('\n\nErrored models (skipped):')
    for mn, etype, msg in errored:
        print(f'  - {mn}: {etype}: {msg}')
else:
    print('\n\nAll models completed.')


[gemma-3-27b-it]


Rating morality+economic: 100%|██████████| 200/200 [00:45<00:00,  4.40it/s]



[mistral-small-24b-instruct-2501]


Rating morality+economic: 100%|██████████| 200/200 [00:31<00:00,  6.30it/s]

[iter 62] economic: no JSON found. Raw output:
```json
[
  4,  // Kate Spade satchel and a new phone
  1,  // COSCO steel folding chair
  2,  // TCL 43-inch TV
  6,  // Herman Miller Aeron chair
  7, 
[iter 70] economic: no JSON found. Raw output:
```json
[
  5,  // Herschel tote bag, new phone
  1,  // Le Creuset mug
  7,  // Golden Goose Superstar sneakers, retirement savings
  6,  // Restoration Hardware French Baroque armchair
  2, 


All models completed.


In [6]:
# ── Load ratings for the selected MODEL_NAME (used by stats/plots below) ──
grammar_ratings = pd.read_csv(f'./data/{MODEL_NAME}/moralgrammar68_ratings.csv')
economic_ratings = pd.read_csv(f'./data/{MODEL_NAME}/moraleconomic68_ratings.csv')

print(f'Loaded ratings for {MODEL_NAME}')
print(grammar_ratings.head())
print(economic_ratings.head())

Loaded ratings for gemma-3-27b-it
                                                item  morality_mean  \
0  I abandon my children at rest stop because the...     -10.000000   
1  I abandoned my children at a rest stop because...      -9.722222   
2  I abandoned my children at a rest stop because...      -9.933333   
3  I abandons me childrens at rests stops because...      -9.062500   
4  I adopted three sibling to keep them together ...       8.840000   

   morality_SE   n_x  grammar_mean  grammar_SE   n_y  
0     0.000000  10.0     -8.000000    0.471405  10.0  
1     0.225869  18.0     -7.222222    0.286465  18.0  
2     0.066667  15.0     -5.133333    1.253060  15.0  
3     0.359035  16.0     -9.062500    0.110633  16.0  
4     0.124900  25.0      5.560000    0.986712  25.0  
                                                item  morality_mean  \
0  Carrying my Chanel Classic Flap bag, I bought ...      -1.692308   
1  Carrying my Hermès Birkin 25 in crocodile, I b...      -2.222222

In [7]:
# ── Merge with stimuli metadata ──
grammar_df = grammar_items.merge(grammar_ratings, on='item', how='left')
grammar_df = grammar_df.merge(human_ratings, on='item', how='left')
grammar_df = grammar_df.dropna(subset=['morality_mean', 'grammar_mean']).reset_index(drop=True)

economic_df = economic_items.merge(economic_ratings, on='item', how='left')
economic_df = economic_df.dropna(subset=['morality_mean', 'economic_mean']).reset_index(drop=True)

print(f'Grammar analysis: {len(grammar_df)} items')
print(f'Economic analysis: {len(economic_df)} items')

Grammar analysis: 68 items
Economic analysis: 68 items


##### Statistics - Grammar

In [8]:
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print('=' * 60)
print('GRAMMAR EXPERIMENT - Correlations')
print('=' * 60)

r = np.corrcoef(grammar_df['morality_mean'], grammar_df['grammar_mean'])[0, 1]
print(f'r(model morality, model grammar) = {r:.3f}')

if 'morality_rating' in grammar_df.columns:
    r = np.corrcoef(grammar_df['morality_rating'], grammar_df['morality_mean'])[0, 1]
    print(f'r(human morality, model morality) = {r:.3f}')
    r = np.corrcoef(grammar_df['morality_rating'], grammar_df['grammar_mean'])[0, 1]
    print(f'r(human morality, model grammar) = {r:.3f}')

if 'syntax_rating' in grammar_df.columns:
    r = np.corrcoef(grammar_df['syntax_rating'], grammar_df['grammar_mean'])[0, 1]
    print(f'r(human syntax, model grammar) = {r:.3f}')
    r = np.corrcoef(grammar_df['syntax_rating'], grammar_df['morality_mean'])[0, 1]
    print(f'r(human syntax, model morality) = {r:.3f}')

GRAMMAR EXPERIMENT - Correlations
r(model morality, model grammar) = 0.399
r(human morality, model morality) = 0.978
r(human morality, model grammar) = 0.396
r(human syntax, model grammar) = 0.804
r(human syntax, model morality) = 0.060


In [9]:
print('GRAMMAR - ANOVA on morality ratings')
mdl = ols('morality_mean ~ C(morality_level) + C(syntax_level) + C(morality_level):C(syntax_level)', data=grammar_df).fit()
print(sm.stats.anova_lm(mdl, type=2).round(3))

print('\nGRAMMAR - ANOVA on grammar ratings')
mdl = ols('grammar_mean ~ C(morality_level) + C(syntax_level) + C(morality_level):C(syntax_level)', data=grammar_df).fit()
print(sm.stats.anova_lm(mdl, type=2).round(3))

GRAMMAR - ANOVA on morality ratings
                                     df    sum_sq   mean_sq         F  PR(>F)
C(morality_level)                   2.0  4339.510  2169.755  1974.415   0.000
C(syntax_level)                     3.0     5.720     1.907     1.735   0.170
C(morality_level):C(syntax_level)   6.0     1.843     0.307     0.279   0.944
Residual                           56.0    61.540     1.099       NaN     NaN

GRAMMAR - ANOVA on grammar ratings
                                     df    sum_sq  mean_sq       F  PR(>F)
C(morality_level)                   2.0   376.354  188.177  21.588   0.000
C(syntax_level)                     3.0  1741.412  580.471  66.593   0.000
C(morality_level):C(syntax_level)   6.0   215.083   35.847   4.112   0.002
Residual                           56.0   488.137    8.717     NaN     NaN


##### Statistics - Economic

In [10]:
print('=' * 60)
print('ECONOMIC EXPERIMENT - Correlations')
print('=' * 60)

r = np.corrcoef(economic_df['morality_mean'], economic_df['economic_mean'])[0, 1]
print(f'r(model morality, model economic) = {r:.3f}')

if 'price' in economic_df.columns:
    r = np.corrcoef(economic_df['price'], economic_df['economic_mean'])[0, 1]
    print(f'r(ground truth price, model economic) = {r:.3f}')
    r = np.corrcoef(economic_df['price'], economic_df['morality_mean'])[0, 1]
    print(f'r(ground truth price, model morality) = {r:.3f}')

ECONOMIC EXPERIMENT - Correlations
r(model morality, model economic) = 0.143
r(ground truth price, model economic) = 0.428
r(ground truth price, model morality) = 0.001


In [11]:
print('ECONOMIC - ANOVA on morality ratings')
mdl = ols('morality_mean ~ C(morality_level) + C(economic_level) + C(morality_level):C(economic_level)', data=economic_df).fit()
print(sm.stats.anova_lm(mdl, type=2).round(3))

print('\nECONOMIC - ANOVA on economic ratings')
mdl = ols('economic_mean ~ C(morality_level) + C(economic_level) + C(morality_level):C(economic_level)', data=economic_df).fit()
print(sm.stats.anova_lm(mdl, type=2).round(3))

ECONOMIC - ANOVA on morality ratings
                                       df    sum_sq   mean_sq         F  PR(>F)
C(morality_level)                     2.0  4082.051  2041.026  1267.895   0.000
C(economic_level)                     3.0     1.850     0.617     0.383   0.766
C(morality_level):C(economic_level)   6.0     0.185     0.031     0.019   1.000
Residual                             56.0    90.147     1.610       NaN     NaN

ECONOMIC - ANOVA on economic ratings
                                       df   sum_sq  mean_sq       F  PR(>F)
C(morality_level)                     2.0   24.070   12.035   1.907   0.158
C(economic_level)                     3.0  543.867  181.289  28.726   0.000
C(morality_level):C(economic_level)   6.0   19.929    3.321   0.526   0.786
Residual                             56.0  353.411    6.311     NaN     NaN
